In [1]:
# ==============================
# CELL 1: IMPORTS & SETUP
# ==============================
import pandas as pd
import numpy as np
import re
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


[nltk_data] Downloading package stopwords to /home/rahul/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# ==============================
# CELL 2: LOAD & PREPARE DATA
# ==============================

# Load datasets
fake = pd.read_csv('../data/Fake.csv')
true = pd.read_csv('../data/True.csv')

# Add labels
fake['label'] = 0   # FAKE
true['label'] = 1   # REAL

# Combine datasets
df = pd.concat([fake, true], axis=0)

# Shuffle dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (44898, 5)


,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1


In [3]:
# ==============================
# CELL 3: FEATURE ENGINEERING
# ==============================

# Combine title and text for better context
df['content'] = df['title'] + " " + df['text']

print("Sample Combined Text:\n")
print(df['content'][0][:500])

Sample Combined Text:

Ben Stein Calls Out 9th Circuit Court: Committed a ‘Coup d’état’ Against the Constitution 21st Century Wire says Ben Stein, reputable professor from, Pepperdine University (also of some Hollywood fame appearing in TV shows and films such as Ferris Bueller s Day Off) made some provocative statements on Judge Jeanine Pirro s show recently. While discussing the halt that was imposed on President Trump s Executive Order on travel. Stein referred to the judgement by the 9th Circuit Court in Washingto


In [4]:
# ==============================
# CELL 4: TEXT PREPROCESSING
# ==============================

stop_words = set(stopwords.words('english'))

def clean(text):
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove punctuation & numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # Tokenize
    words = text.split()

    # Remove stopwords
    words = [w for w in words if w not in stop_words]

    return ' '.join(words)

# Apply cleaning
df['content'] = df['content'].apply(clean)

print("✅ Text preprocessing completed")

✅ Text preprocessing completed


In [5]:
# ==============================
# CELL 5: SPLIT & VECTORIZATION
# ==============================

X = df['content']
y = df['label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# TF-IDF Vectorizer
vectorizer = TfidfVectorizer(
    max_features=70000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Vectorization completed")
print("Train shape:", X_train_vec.shape)

Vectorization completed
Train shape: (35918, 70000)


In [7]:
# ==============================
# CELL 6: MODEL TRAINING & SAVE
# ==============================

# Train model
model = LogisticRegression(max_iter=2000)
model.fit(X_train_vec, y_train)

# Predictions
y_pred = model.predict(X_test_vec)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Save model
with open('../data/best_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('../data/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("\n✅ Model & Vectorizer saved successfully")

Accuracy: 0.9880846325167038

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4710
           1       0.98      0.99      0.99      4270

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980


✅ Model & Vectorizer saved successfully
